# Entrenamiento: Clústering y regresión con redes neuronales

**Propósito**: Entrenar y evaluar modelos de regresión basados en redes neuronales (MLP) utilizando tanto un enfoque híbrido (clústering + MLP en cada grupo) como un modelo general, comparando sus resultados para obtener el mejor.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, pairwise_distances_argmin_min
from sklearn.base import BaseEstimator, RegressorMixin

from sklearn.neural_network import MLPRegressor
import warnings

#Importar utilidades de TFM
sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import utils as tfm_utils


In [2]:
#Inicializar sesión de Spark
spark=tfm_utils.get_spark_session(app_name="ML_training")

#Cargar los datos desde Delta Lake
df_spark=tfm_utils.read_from_delta(spark, "ml.idealista_crem.dataset_for_analysis")
df=df_spark.toPandas()
print(f"Dataset cargado con éxito. Dimensiones: {df.shape}")

#Definimos la semilla de aleatoriedad que utilizaremos
random_state=55

Dataset cargado con éxito. Dimensiones: (10509, 22)


## Clustering.

Lo primero que haremos será dividir los datos de origen en dos grupos, aquellos con los que entrenaremos el modelo y aquellos que nos servirán para testear los modelos obtenidos, nos quedaremos un un 85% de los datos para testo.

In [3]:
#Definir variables independientes (X) y dependiente (y)
X=df.drop(columns=['price'])
y=df['price']

#Dividir en conjuntos de entrenamiento y test (85% train, 15% test)
X_train, X_test, y_train, y_test=train_test_split(
    X, y, test_size=0.15, random_state=random_state
)

print(f"Entrenamiento: {X_train.shape[0]} muestras")
print(f"Testeo: {X_test.shape[0]} muestras")

Entrenamiento: 8932 muestras
Testeo: 1577 muestras


Aplicamos DBSCAN para obtener grupos en los datos. Hay que decir que para hacer los grupos, o clusters usaremos solo las variables características, esto es importante. Lo que queremos predecir es el precio de un inmueble a partir de sus características, si usaramos el precio para hacer ese clústering, además de no tener sentido con lo que queremos hacer, estaríamos comentiendo un sesgo respecto esa variable, los grupos se llevarían a cabo respecto a dicho valor. Así mismo, la división entre dataset de entrenamiento y de testeo la hacemos antes de entrenar cualquier modelo de regresión. Cuando apliquemos un testeo vamos a querer observar el comportamiento de todo el proceso predictivo, no solo de una parte de él.

In [4]:
#Creamos un modelo DBSCAN con eps=2, el radio del vecindario,
#y min_samples=15, la cantidad de vecinos para pertenecer a un grupo
dbscan = DBSCAN(eps=2, min_samples=15)

#Ajustamos para obtener los clusters
cluster_labels = dbscan.fit_predict(X_train)

#Recuperamos las etiquetas a los datos de entrenamiento
X_train['cluster']=cluster_labels

#Veamos cuantos registros hay por grupo
display(X_train.groupby("cluster").size().reset_index(name="n_observaciones"))

,cluster,n_observaciones
0,-1,74
1,0,5275
2,1,2306
3,2,1277


Ahora bien, el problema con este modelo es que podemos crear los clústeres a partir de un subconjunto y luego, con muestras nuevas, no podemos predecir a qué grupo pertenecen dichas muestras, al menos no con un método directo, lo que sí podemos hacer es construir una estrategia con la cual, crear los clusters a partir de nuestros datos de entrenamiento y usar el resultado para asignar a a un grupo una nueva muestra.

In [5]:
def predict_dbscan(dbscan_model, X_new):
    """
    Asigna nuevas muestras (X_new) a los clusteres DBSCAN ya entrenados.
    
    Estrategia:
    Asigna cada nueva muestra al cluster de su punto nucleo mas cercano si la distancia es menor o igual a 'eps'. De lo contrario, la clasifica como ruido (-1).
    """
    #Si no se formaron puntos nucleo durante el entrenamiento, todo nuevo punto es ruido (-1)
    if len(dbscan_model.core_sample_indices_)==0:
        return np.full(X_new.shape[0], -1)
    
    #Extraemos los puntos nucleo de los grupos
    core_samples=dbscan_model.components_
    
    #Extraemos las etiquetas de los nucleos
    core_labels=dbscan_model.labels_[dbscan_model.core_sample_indices_]
    
    #Calculamos la distancia al nucleo mas cercano para cada muestra
    #pairwise_distances_argmin_min devuelve:
    #    - index: El indice del punto nucleo mas cercano.
    #    - dist: La distancia euclidiana hacia ese punto 
    #           nucleo mas cercano.
    closest_core_idx, distances=pairwise_distances_argmin_min(X_new, core_samples)
    
    #Aplicamos la regla de asignacion:
    #    Si dist <= eps: se le asigna la etiqueta de ese punto nucleo.
    #    Si dist > eps: se le asigna -1 (ruido).
    predicted_labels = np.where(
        distances <= dbscan_model.eps, 
        core_labels[closest_core_idx], 
        -1
    )
    
    return predicted_labels

X_test['cluster']=predict_dbscan(dbscan, X_test)

#Veamos cuantos registros hay por grupo
display(X_test.groupby("cluster").size().reset_index(name="n_observaciones"))

,cluster,n_observaciones
0,-1,14
1,0,912
2,1,430
3,2,221


Reconstruimos los dataset de entrenamiento y testeo con su precio para poder trabajar con los datos de mejor manera de forma posterior.

In [6]:
df_train_full = X_train.copy()
df_train_full['price'] = y_train

df_test_full = X_test.copy()
df_test_full['price'] = y_test

Definimos una función con la cual entrenar un modelo de regresión basado en redes neuronales (MLPRegressor) y optimizar sus hiperparámetros, para decidir cuál es la mejor configuración utilizaremos la métrica del error medio en valor absoluto. Para el entrenamiento de estos modelos vamos a asumir que, como resultado del clústering anterior, los datos de cada clúster se parecen lo suficiente entre sí, precisamente por eso hemos escogido dicho algoritmo de clústering, sin embargo, ahora, para entrenar los modelos de regresión, sí que tenemos a nuestra disposición la variable de precios, por lo que, lo primero que haremos, será eliminar los outliers locales de cada conjunto respecto a esta nueva variable. Once hecho esto entraremos en el entrenamiento del modelo de red neuronal para dicho clúster.

In [7]:
def delete_outliers_iqr(df, columna='price', factor=1.5):
    """
    Elimina outliers usando el método IQR
    """
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - factor * IQR
    limite_superior = Q3 + factor * IQR
    
    #Filtrar datos
    df_filtrado = df[(df[columna] >= limite_inferior) & 
                     (df[columna] <= limite_superior)]
    
    print(f"Registros eliminados: {len(df) - len(df_filtrado)} ({100*(1-len(df_filtrado)/len(df)):.1f}%)")
    print(f"Límites: [{limite_inferior:.2f}, {limite_superior:.2f}]")
    
    return df_filtrado

In [8]:
warnings.filterwarnings('ignore')

def get_best_local_model(df_full, cluster_label):
    #Filtramos los datos del cluster a procesar
    df_local=df_full[df_full['cluster']==cluster_label].copy()

    #Eliminamos los outliers respecto del precio
    df_local=delete_outliers_iqr(df_local)
        
    #Definir variables independientes (X) y dependiente (y)
    X_local=df_local.drop(columns=['price','cluster'])
    y_local=df_local['price']
    
    #Dividir en conjuntos de entrenamiento y test (85% train, 15% test)
    X_local_train, X_local_test, y_local_train, y_local_test=train_test_split(X_local, y_local, test_size=0.15
                                                                              , random_state=random_state)

    #Modelo base de red neuronal (MLPRegressor)
    mlp_base = MLPRegressor(
        max_iter=150,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=random_state
    )

    param_grid = {
        'hidden_layer_sizes': [(64, 32), (32, 16)],
        'alpha': [0.0001, 0.001],
        'learning_rate_init': [0.001, 0.01]
    }
       
    #Nos quedamos con el modelo con menor error medio absoluto
    grid_search = GridSearchCV(
        estimator=mlp_base,
        param_grid=param_grid,
        cv=3,
        scoring='neg_mean_absolute_error',
        n_jobs=-1,
        verbose=0
    )
    grid_search.fit(X_local_train, y_local_train)
    
    best_params = grid_search.best_params_
    best_model = grid_search.best_estimator_
    
    print(f"Mejores parámetros: {best_params}")
    
    #Evaluar el mejor modelo en el conjunto de validación local
    y_pred_train=best_model.predict(X_local_train)
    y_pred_test=best_model.predict(X_local_test)

    mae_train = mean_absolute_error(y_local_train, y_pred_train)
    mae_test = mean_absolute_error(y_local_test, y_pred_test)
    
    print(f"MAE Train: {mae_train:.2f} €")
    print(f"MAE Test: {mae_test:.2f} €")

    return best_model

In [9]:
get_best_local_model(df_train_full, 0)

Registros eliminados: 232 (4.4%)
Límites: [-120062.50, 512037.50]
Mejores parámetros: {'alpha': 0.001, 'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.01}
MAE Train: 52043.83 €
MAE Test: 53592.39 €


,loss,'squared_error'
,hidden_layer_sizes,"(64, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.01
,power_t,0.5
,max_iter,150
,shuffle,True


In [10]:
get_best_local_model(df_train_full, 1)

Registros eliminados: 114 (4.9%)
Límites: [-155000.00, 605000.00]
Mejores parámetros: {'alpha': 0.001, 'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.01}
MAE Train: 73059.02 €
MAE Test: 75706.13 €


,loss,'squared_error'
,hidden_layer_sizes,"(64, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.01
,power_t,0.5
,max_iter,150
,shuffle,True


In [11]:
get_best_local_model(df_train_full, 2)

Registros eliminados: 76 (6.0%)
Límites: [-177500.00, 626500.00]
Mejores parámetros: {'alpha': 0.0001, 'hidden_layer_sizes': (64, 32), 'learning_rate_init': 0.01}
MAE Train: 71257.80 €
MAE Test: 64074.59 €


,loss,'squared_error'
,hidden_layer_sizes,"(64, ...)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.01
,power_t,0.5
,max_iter,150
,shuffle,True
